<a href="https://colab.research.google.com/github/Nanda-Lopes/AlgoStudies/blob/main/Stanford_Algorithms_Specialization_Course3_W2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clustering de Espaçamento Máximo

O objetivo é dividir um grafo de $n$ vértices em exatamente $k = 4$ grupos (clusters) maximizando a distância de separação (espaçamento) entre eles. O espaçamento é definido como a menor distância entre quaisquer dois nós que pertençam a clusters distintos.

O algoritmo guloso que resolve este problema herda a estrutura do Algoritmo de Kruskal para Árvores Geradoras Mínimas (MST):
1. Cada vértice foi tratado individualmente como um componente único usando um *Union-Find*.
2. Todas as arestas foram ordenadas de forma crescente de distância.
3. Caso os vértices adjacentes pertencessem a clusters diferentes, os mesmos foram unidos.
4. As fusões foram paradas no exato momento em que o número total de clusters atingiu $k = 4$.
5. A partir desse ponto, a próxima aresta da lista que conectasse componentes distintos indicaria a menor distância de fronteira remanescente, sendo este peso o espaçamento ótimo máximo.

In [3]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n + 1))
        self.rank = [0] * (n + 1)
        self.count = n

    def find(self, i):
        if self.parent[i] == i:
            return i
        self.parent[i] = self.find(self.parent[i])
        return self.parent[i]

    def union(self, i, j):
        root_i = self.find(i)
        root_j = self.find(j)
        if root_i != root_j:
            if self.rank[root_i] < self.rank[root_j]:
                self.parent[root_i] = root_j
            elif self.rank[root_i] > self.rank[root_j]:
                self.parent[root_j] = root_i
            else:
                self.parent[root_j] = root_i
                self.rank[root_i] += 1
            self.count -= 1
            return True
        return False

filename = "clustering1.txt"
edges = []
num_nodes = 0

with open(filename, "r") as f:
    lines = f.readlines()
    num_nodes = int(lines[0].strip())
    for line in lines[1:]:
        if line.strip():
            u, v, cost = map(int, line.split())
            edges.append((cost, u, v))

edges.sort()
uf = UnionFind(num_nodes)

for cost, u, v in edges:
    if uf.count > 4:
        uf.union(u, v)
    else:
        if uf.find(u) != uf.find(v):
            print("==================================")
            print("RESPOSTA QUESTÃO 1 (MAX SPACING):")
            print(cost)
            print("==================================")
            break

RESPOSTA QUESTÃO 1 (MAX SPACING):
106


# Clustering Implícito com Distância de Hamming

Neste problema, o conjunto de dados é massivo ($n = 200.000$ nós, representados por sequências de 24 bits). O custo de uma aresta entre dois nós é a Distância de Hamming (o número de posições em que os bits diferem).

Desejamos calcular o maior número de clusters $k$ de forma que o espaçamento seja no mínimo $3$. Isso significa que quaisquer dois nós que diferem em menos de 3 bits (distâncias 0, 1 ou 2) devem necessariamente ser agrupados no mesmo cluster.

Como um algoritmo quadrático tradicional $O(n^2)$ falharia devido ao tamanho da entrada, uma solução altamente performática foi implementada com base em Tabela Hash e operações binárias bit a bit:
1. Os rótulos de bits foram mapeados de binário para inteiros e salvos em uma tabela hash (label_to_node). Esse mapeamento agrupa automaticamente todas as duplicatas exatas (distância de Hamming $0$).
2. Para cada um dos nós com rótulo único, foram geradas todas as 24 variações possíveis de distância 1, invertendo um único bit (via operador XOR ^ com deslocamento de bits 1 << i). Cada vizinho virtual, caso existisse, foram unidos.
3. Todas as $\binom{24}{2} = 276$ combinações possíveis de distância 2 (invertendo 2 bits em cada nó via XOR duplo) foram geradas de maneira idêntica, com suas uniões equivalentes efetuadas.
4. O resultado final do número de partições válidas do Union-Find após essas fusões representa a quantidade máxima de clusters conexos com espaçamento estritamente controlado.

In [4]:
class UnionFindBig:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
        self.count = n

    def find(self, i):
        if self.parent[i] == i:
            return i
        self.parent[i] = self.find(self.parent[i])
        return self.parent[i]

    def union(self, i, j):
        root_i = self.find(i)
        root_j = self.find(j)
        if root_i != root_j:
            if self.rank[root_i] < self.rank[root_j]:
                self.parent[root_i] = root_j
            elif self.rank[root_i] > self.rank[root_j]:
                self.parent[root_j] = root_i
            else:
                self.parent[root_j] = root_i
                self.rank[root_i] += 1
            self.count -= 1
            return True
        return False

filename = "clustering_big.txt"
label_to_node = {}
num_nodes = 0
num_bits = 0

with open(filename, "r") as f:
    header = f.readline()
    num_nodes, num_bits = map(int, header.split())
    for idx, line in enumerate(f):
        if line.strip():
            val = int(line.replace(" ", "").strip(), 2)
            if val not in label_to_node:
                label_to_node[val] = idx

uf = UnionFindBig(num_nodes)

with open(filename, "r") as f:
    f.readline()
    for idx, line in enumerate(f):
        if line.strip():
            val = int(line.replace(" ", "").strip(), 2)
            first_occurrence = label_to_node[val]
            uf.union(idx, first_occurrence)

for val, node_idx in label_to_node.items():
    for i in range(num_bits):
        neighbor = val ^ (1 << i)
        if neighbor in label_to_node:
            uf.union(node_idx, label_to_node[neighbor])

for val, node_idx in label_to_node.items():
    for i in range(num_bits):
        for j in range(i + 1, num_bits):
            neighbor = val ^ (1 << i) ^ (1 << j)
            if neighbor in label_to_node:
                uf.union(node_idx, label_to_node[neighbor])

print("==================================")
print("RESPOSTA QUESTÃO 2 (MAX CLUSTERS):")
print(uf.count)
print("==================================")

RESPOSTA QUESTÃO 2 (MAX CLUSTERS):
6118
